# Jake's Weekly Stack — one notebook, whole routine
**Run order** — *Monday pre-open:* 1→2→3→4. *Friday evening:* 3→5→6→7. *Event-driven:* 8 (insider detail on tripwires), 9 (options mark), 10 (ignition filter — panic windows).

Everything is manual — no scheduling, nothing runs unattended. Paste summary blocks back to chat for the vault log.
Each cell is self-contained except `EMAIL` (config below).

In [ ]:
EMAIL = "your_email@example.com"   # <- REAL email (SEC EDGAR requires it)
%pip -q install yfinance
print("config ok:", EMAIL)

## 1 · Monday flows — where money went last week (tape: direction × conviction)

In [ ]:
# MONDAY FLOWS — where money went last week (manual pre-open Monday run; pairs with
# structural_pulls.py: run this -> COT cell -> Form 4 tripwire. ~20 min total.)
# Built 2026-07-11. No cron per standing spending rule.
import yfinance as yf, pandas as pd, numpy as np

SECTORS = {"XLK":"tech","SMH":"semis","XLE":"energy","XLV":"health","XLF":"fins","XLI":"indus",
           "XLP":"staples","XLY":"discret","XLU":"utes","XLB":"materials","XLRE":"REITs",
           "XLC":"comms","GLD":"gold","USO":"crude","TLT":"20y bond","UUP":"dollar","IBIT":"btc"}
rows = []
for t, name in SECTORS.items():
    h = yf.Ticker(t).history(period="4mo")
    if h.empty: continue
    dv = h.Close * h.Volume
    rows.append({"etf": t, "sector": name,
        "1w%": round((h.Close.iloc[-1]/h.Close.iloc[-6]-1)*100, 1),
        "1m%": round((h.Close.iloc[-1]/h.Close.iloc[-22]-1)*100, 1),
        "$vol_1w_vs_3m": round(dv.iloc[-5:].mean()/dv.iloc[-65:-5].mean(), 2)})
d = pd.DataFrame(rows).sort_values("1w%", ascending=False)
print(d.to_string(index=False))
print("\nREAD: 1w% = direction | $vol ratio = conviction (>1.3 = money moving).")

import requests
from io import StringIO
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=30).text
tickers = [str(t).replace(".", "-") for t in pd.read_html(StringIO(html))[0]["Symbol"]]
px = yf.download(tickers, period="4mo", auto_adjust=True, progress=False)
close, vol = px["Close"], px["Volume"]
dv = close * vol
ret1w = (close.iloc[-1]/close.iloc[-6]-1)*100
flow = (dv.iloc[-5:].mean()/dv.iloc[-65:-5].mean()).replace([np.inf], np.nan)
f = pd.DataFrame({"1w%": ret1w.round(1), "flow_x": flow.round(2)}).dropna()
hot = f[f.flow_x > 1.5].copy()
hot["signed"] = hot.flow_x * np.sign(hot["1w%"])
ins, outs = hot[hot["1w%"] > 0], hot[hot["1w%"] < 0]
print(f"MONEY IN (abnormal volume, up) — {len(ins)} names:")
print(ins.nlargest(12, "flow_x").to_string() if len(ins) else "  NONE — no abnormal-volume accumulation this week (itself a signal: drift, not buying)")
print(f"\nMONEY OUT (abnormal volume, down) — {len(outs)} names:")
print(outs.nlargest(12, "flow_x").to_string() if len(outs) else "  NONE")


## 2 · COT — leveraged funds (ES/NQ) + managed money (CL/GC). Friday 3:30pm ET release, Tuesday data.
Levels are structural (basis trades); **deltas are the signal**. Crowding = stored reversal energy.

In [ ]:
import pandas as pd, requests, time
from datetime import date, timedelta

# ---------------- CELL A (v4: params-encoded — fixes S&P ampersand URL split) ----------------
def cot(dataset, mkt, prefix, label):
    r = requests.get(f"https://publicreporting.cftc.gov/resource/{dataset}.json",
                     params={"$where": f"contract_market_name='{mkt}'",
                             "$order": "report_date_as_yyyy_mm_dd DESC", "$limit": 8}, timeout=30)
    rows = r.json()
    if not rows: print(f"{label}: no rows"); return
    d = pd.DataFrame(rows)
    lc = [c for c in d.columns if prefix in c and "long" in c and not any(x in c for x in ("spread","change","pct"))]
    sc = [c for c in d.columns if prefix in c and "short" in c and not any(x in c for x in ("spread","change","pct"))]
    if not (lc and sc): print(f"{label}: no {prefix} cols"); return
    d["net"] = d[lc[0]].astype(float) - d[sc[0]].astype(float)
    cur, prior = d.net.iloc[0], d.net.iloc[min(4, len(d)-1)]
    print(f"{label} [{mkt}] {d.report_date_as_yyyy_mm_dd.iloc[0][:10]}: net {cur:+,.0f} | 4wk {prior:+,.0f} | delta {cur-prior:+,.0f}")

cot("gpe5-46if", "E-MINI S&P 500", "lev_money", "ES levfunds")
cot("gpe5-46if", "S&P 500 Consolidated", "lev_money", "SPX consol levfunds")
cot("gpe5-46if", "NASDAQ MINI", "lev_money", "NQ levfunds")
cot("72hh-3qpy", "CRUDE OIL, LIGHT SWEET-WTI", "m_money", "CL mgd-money")
cot("72hh-3qpy", "GOLD", "m_money", "GC mgd-money")
# Sector texture (thin volume, read direction only): E-MINI S&P TECHNOLOGY/ENERGY/FINANCIAL/HEALTH CARE INDEX



## 3 · Top-10 composite vs rolling medians — the regime line (validation gates still pending)

In [ ]:
import yfinance as yf, pandas as pd
TOP10 = ["NVDA","MSFT","AAPL","AMZN","META","AVGO","GOOGL","TSLA","LLY","JPM"]  # edit to current top-10 by weight
px = yf.download(TOP10, period="2y", auto_adjust=True, progress=False)["Close"].dropna()
comp = px.mean(axis=1)
for w in (50, 200):
    med = comp.rolling(w).median()
    state = "ABOVE" if comp.iloc[-1] > med.iloc[-1] else "BELOW"
    print(f"{w}d median: comp {comp.iloc[-1]:.1f} vs {med.iloc[-1]:.1f} -> {state}"
          f" | distance {(comp.iloc[-1]/med.iloc[-1]-1)*100:+.1f}%")
print("jurisdiction: median line governs the CORE sleeve pace only; thesis/falsifiers govern the hedge.")

## 4 · Form 4 tripwire — filing counts, 14d, all vault watchlists (counts ≠ direction; 3+ → run cell 8)

In [ ]:
import pandas as pd, requests, time
from datetime import date, timedelta

# ---------------- CELL B ----------------
# EMAIL set in the config cell
H = {"User-Agent": f"Jake research {EMAIL}"}
WATCH = {
 "semi_grid": ["MU","LRCX","AMAT","NVDA","AMD","AVGO","KLAC","TER","ON","MRVL","MPWR","SMCI","WDC","STX","SNDK"],
 "quiet_health": ["PG","GILD","CME","GD","HIG","EA","FFIV","ULTA","GEHC","CBOE","UHS","WRB","CB","HII","CI"],
 "mold_12": ["VRRM","GTM","DUOL","INSP","EPAM","FRPT","BLKB","AMPH","GPK","ADMA","UPWK","WEN"],
 "bottleneck": ["CLF","VICR","VSH","ENS","VECO","AXTI","POWL","STRL","FIX","TTMI"],
}
cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}
SINCE = (date.today() - timedelta(days=14)).isoformat()

def sweep(t):
    c = cik.get(t)
    if not c: return None
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{c}.json", headers=H, timeout=30).json()
    r = sub["filings"]["recent"]; time.sleep(0.12)
    return sum(1 for i in range(len(r["form"])) if r["form"][i] == "4" and r["filingDate"][i] >= SINCE)

for group, names in WATCH.items():
    hits = {t: sweep(t) for t in names}
    flagged = {t: n for t, n in hits.items() if n}
    print(f"{group}: {flagged if flagged else 'no Form 4s in 14d'}")



## 5 · Short interest snapshot (bi-monthly data — read the MoM change)

In [ ]:
# ---------------- CELL C ----------------
import yfinance as yf
for t in ["MU","NVDA","GPK","FRPT","DUOL","UPWK","VECO","AXTI","ORCL"]:
    i = yf.Ticker(t).info
    spf = i.get("shortPercentOfFloat"); ss, ssp = i.get("sharesShort"), i.get("sharesShortPriorMonth")
    d = f"{(ss/ssp-1)*100:+.0f}% MoM" if ss and ssp else "n/a"
    print(f"{t}: short {spf*100:.1f}% of float | {d}" if spf else f"{t}: n/a")



## 6 · 13F freshness — tracked managers (quarterly, 45d lag; Scion dark since Nov-25)

In [ ]:
import pandas as pd, requests, time
from datetime import date, timedelta

# ---------------- CELL D ----------------
MANAGERS = {"Scion (Burry)": "0001649339", "Berkshire": "0001067983", "Appaloosa": "0001656456"}
for name, c in MANAGERS.items():
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{c.zfill(10)}.json", headers=H, timeout=30).json()
    r = sub["filings"]["recent"]; time.sleep(0.12)
    f13 = [(r["filingDate"][i], r["accessionNumber"][i]) for i in range(len(r["form"])) if "13F" in r["form"][i]]
    if f13:
        print(f"{name}: latest 13F {f13[0][0]} https://www.sec.gov/Archives/edgar/data/{int(c)}/{f13[0][1].replace('-','')}")
    else:
        print(f"{name}: none")


## 7 · *(spare)*

In [ ]:
# scratch cell


## 8 · Dense insider pull — full Form 4 detail on flagged names
Codes: P=open-mkt buy (highest signal), S=sell, M+S=comp monetization, A/F/G=noise. 10b5-1 column auto-separates scheduled sells. 3+ distinct P-buyers = CLUSTER flag.

In [ ]:
# DENSE INSIDER PULL — full Form 4 detail from EDGAR (primary source), any tickers, any window.
# Built 2026-07-11 consolidating the scattered chat cells. Per filing: owner, title, transaction
# code, $ value, and the 10b5-1 checkbox (auto-separates scheduled comp sales from discretionary).
# Cluster read: distinct discretionary buyers in window = the MTDR-signature detector.
import requests, time, re
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import date, timedelta

# EMAIL set in the config cell
H = {"User-Agent": f"Jake research {EMAIL}"}
TICKERS = ["ENS", "VICR", "TTMI", "CME", "HII", "GTM"]   # <- edit
DAYS = 30                                          # lookback window
SINCE = (date.today() - timedelta(days=DAYS)).isoformat()

CODES = {"P": "BUY (open mkt)", "S": "SELL", "A": "grant/award", "M": "option exercise",
         "F": "tax withholding", "G": "gift", "D": "disposition to issuer", "C": "conversion"}

cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}

def txt(el, path):
    e = el.find(path)
    return e.text.strip() if e is not None and e.text else None

all_rows = []
for t in TICKERS:
    c = cik.get(t)
    if not c:
        print(f"{t}: no CIK"); continue
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{c}.json", headers=H, timeout=30).json()
    time.sleep(0.12)
    rec = sub["filings"]["recent"]
    f4s = [(rec["accessionNumber"][i], rec["filingDate"][i], rec["primaryDocument"][i])
           for i in range(len(rec["form"]))
           if rec["form"][i] == "4" and rec["filingDate"][i] >= SINCE]
    for acc, fdate, pdoc in f4s:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(c)}/{acc.replace('-','')}/{pdoc.split('/')[-1]}"
        try:
            x = requests.get(url, headers=H, timeout=30).text
            time.sleep(0.12)
            root = ET.fromstring(re.sub(r"<\?xml[^>]*\?>", "", x, count=1))
        except Exception as e:
            print(f"  {t} {acc}: parse fail {str(e)[:40]}"); continue
        owner = txt(root, ".//reportingOwner/reportingOwnerId/rptOwnerName") or "?"
        rel = root.find(".//reportingOwner/reportingOwnerRelationship")
        title = (txt(rel, "officerTitle") or
                 ("Director" if txt(rel, "isDirector") in ("1", "true") else "") or
                 ("10%+ owner" if txt(rel, "isTenPercentOwner") in ("1", "true") else "?"))
        plan = txt(root, ".//aff10b5One") in ("1", "true")   # 10b5-1 checkbox (2023+)
        for tr in root.findall(".//nonDerivativeTransaction"):
            code = txt(tr, "transactionCoding/transactionCode")
            sh = txt(tr, "transactionAmounts/transactionShares/value")
            pr = txt(tr, "transactionAmounts/transactionPricePerShare/value")
            ad = txt(tr, "transactionAmounts/transactionAcquiredDisposedCode/value")
            if not (code and sh):
                continue
            all_rows.append({"ticker": t, "filed": fdate, "owner": owner[:26], "title": title[:24],
                             "code": code, "what": CODES.get(code, code), "A/D": ad,
                             "shares": float(sh), "price": float(pr) if pr else None,
                             "value_$": round(float(sh) * float(pr or 0)),
                             "10b5-1": "YES" if plan else ""})
    print(f"{t}: {len(f4s)} Form 4s parsed")

df = pd.DataFrame(all_rows)
if len(df):
    print("\n=== FULL DETAIL ===")
    print(df.sort_values(["ticker", "filed"]).to_string(index=False))
    print("\n=== SUMMARY (discretionary only — 10b5-1 excluded) ===")
    disc = df[df["10b5-1"] != "YES"]
    for t in df.ticker.unique():
        d = disc[disc.ticker == t]
        buys = d[d.code == "P"]; sells = d[d.code == "S"]
        planned = df[(df.ticker == t) & (df["10b5-1"] == "YES") & (df.code == "S")]["value_$"].sum()
        b_sum, s_sum, n_buyers = buys["value_$"].sum(), sells["value_$"].sum(), buys.owner.nunique()
        cluster = " <-- CLUSTER" if n_buyers >= 3 else ""
        print(f"{t}: disc buys ${b_sum:,} ({n_buyers} distinct buyers{cluster}) | "
              f"disc sells ${s_sum:,} | scheduled 10b5-1 sells ${planned:,}")
else:
    print("no transactions in window")


## 9 · Options chain + book marker (SPY Dec-745P) — delayed ~15min; straddle = the market's event pricing

In [ ]:
import yfinance as yf, pandas as pd
TICKER, BAND = "SPY", 0.06
tk = yf.Ticker(TICKER)
spot = tk.history(period="1d")["Close"].iloc[-1]
print(f"{TICKER} spot ~{spot:.2f} | expirations: {list(tk.options)[:12]} ...")
EXP = tk.options[0]
ch = tk.option_chain(EXP)
c, p = ch.calls, ch.puts
atmC = c.iloc[(c.strike - spot).abs().argmin()]; atmP = p.iloc[(p.strike - spot).abs().argmin()]
strad = (atmC.bid + atmC.ask)/2 + (atmP.bid + atmP.ask)/2
print(f"ATM straddle {EXP}: ~${strad:.2f} = ±{strad/spot*100:.2f}% expected move by expiry")
for e in tk.options:
    if e.startswith("2026-12-18"):
        row = tk.option_chain(e).puts.query("strike == 745.0")
        if len(row):
            r = row.iloc[0]
            print(f"BOOK: SPY {e} 745P: {r.bid}/{r.ask} last {r.lastPrice} | OI {r.openInterest} | IV {r.impliedVolatility*100:.1f}%")

## 10 · Ignition filter — intact wreckage (5 gates). Short list at highs = normal; re-run INTO panics.

In [ ]:
# IGNITION FILTER — "intact wreckage" (strict mold from runner_anatomy, 2026-07-10)
# Gates: dd<=-50% vs 2y high | mcap $0.2-10B | 0<PE<15 | FCF+ | revenue growing
# Necessary-profile, NOT sufficient: the study conditioned on winners; value traps
# pass too. Name-level "is the business intact" work required per survivor.
# Re-run DURING panic windows — that's when the list gets long and the study says
# runners are born. Tiebreaker: insider cluster buys (OpenInsider cell).
import pandas as pd, requests, yfinance as yf
from io import StringIO

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
tk = []
for u in ["List_of_S%26P_500_companies", "List_of_S%26P_400_companies", "List_of_S%26P_600_companies"]:
    html = requests.get(f"https://en.wikipedia.org/wiki/{u}", headers=UA, timeout=30).text
    tk += [str(t).replace(".", "-").strip() for t in pd.read_html(StringIO(html))[0]["Symbol"]]
tk = sorted(set(tk))

px = yf.download(tk, period="2y", auto_adjust=True, progress=False)["Close"].dropna(axis=1, how="all")
dd = px.iloc[-1] / px.max() - 1
crushed = dd[dd <= -0.50].sort_values()            # GATE 1 — the one that matters most
print(f"{len(crushed)} names >=50% below 2y high\n")

rows = []
for t in crushed.index:                             # .info only for the crushed few
    try:
        i = yf.Ticker(t).info
    except Exception:
        continue
    mc, fcf = i.get("marketCap"), i.get("freeCashflow")
    rows.append({"ticker": t, "dd%": round(crushed[t] * 100),
                 "mcap_B": round((mc or 0) / 1e9, 1),
                 "PE": i.get("trailingPE"),
                 "FCFy%": round(fcf / mc * 100, 1) if (fcf and mc) else None,
                 "revG%": round(i["revenueGrowth"] * 100, 1) if i.get("revenueGrowth") is not None else None,
                 "sector": i.get("sector")})
df = pd.DataFrame(rows)

mold = df[df["mcap_B"].between(0.2, 10) & df["PE"].between(0, 15)
          & (df["FCFy%"] > 0) & (df["revG%"] > 0)]
print("=== STRICT MOLD (all 5 gates) ===")
print(mold.sort_values("dd%").to_string(index=False))
print("\n=== NEAR MISSES (crushed + profitable, failed one gate) ===")
near = df[df["mcap_B"].between(0.2, 10) & df["PE"].between(0, 20)].drop(mold.index, errors="ignore")
print(near.sort_values("dd%").to_string(index=False))


---
*Lag-vs-lead key: tape=coincident (volume-backed moves persist) · COT=contrarian at extremes · insiders=leading (months) · short interest=fuel · 13F=stale · mechanical/rule-forced flows=truly predictive. The distribution chain (insiders→HFs→retail) is the master sequence.*